In [1]:
import sys, os, json, torch, math
import torch.nn as nn
from torch.utils.data import DataLoader
import warnings

warnings.filterwarnings('ignore')
# Добавляем корень проекта для правильных импортов
sys.path.append(os.path.abspath('../../..'))

In [2]:
from DL.models.rotate_net import RotateNet
from DL.models.cascade_net import CascadeNet
from DL.trainers.model_trainer import ModelTrainer
from DL.visualization.plot_utils import plot_confusion_matrices
from DL.visualization.classic_gradcam import visualize_classic_gradcam
from DL.data.rotated_cyrillic_dataset import RotatedCyrillicDataset

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


In [4]:
train_dataset = RotatedCyrillicDataset(60000, 42, max_angle=150)
test_dataset = RotatedCyrillicDataset(15000, 999, max_angle=150)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
# 1. Обучение RotateNet
config_path_rot = '../../../DL/configs/rotatenet/rotatenet_flatten.json'
with open(config_path_rot, 'r') as f:
    config_rot = json.load(f)

print("=== ЭТАП 1: Обучение RotateNet (~212 KB) ===")
rotatenet = RotateNet(config_rot).to(device)
opt_rot = torch.optim.Adam(rotatenet.parameters(), lr=1e-3)
mse_loss = nn.MSELoss()

for epoch in range(10):
    rotatenet.train()
    running_loss = 0.0
    for img, _, sin_cos_target in train_loader:
        img, target = img.to(device), sin_cos_target.to(device)
        opt_rot.zero_grad()
        pred_sin_cos, _ = rotatenet(img)
        loss = mse_loss(pred_sin_cos, target)
        loss.backward()
        opt_rot.step()
        running_loss += loss.item() * img.size(0)
    print(f"Эпоха {epoch+1}/10 | MSE Loss: {running_loss / len(train_dataset):.4f}")

=== ЭТАП 1: Обучение RotateNet (~212 KB) ===
Эпоха 1/10 | MSE Loss: 0.4655
Эпоха 2/10 | MSE Loss: 0.2623
Эпоха 3/10 | MSE Loss: 0.1698
Эпоха 4/10 | MSE Loss: 0.1105
Эпоха 5/10 | MSE Loss: 0.0953


In [ ]:
# 2. Обучение Каскада
print("\n=== ЭТАП 2: Обучение 8KB CascadeNet ===")
with open('../../../DL/configs/lenet/lenet_8kb.json', 'r') as f:
    config_8kb = json.load(f)
config_8kb['num_classes'] = len(train_dataset.char_to_idx)

In [ ]:
cascade_model = CascadeNet(config_8kb, rotatenet).to(device)
optimizer_cascade = torch.optim.Adam(cascade_model.classifier.parameters(), lr=1e-3)

trainer_cascade = ModelTrainer(
    cascade_model, optimizer_cascade, nn.CrossEntropyLoss(), device,
    use_background_loss=True, bg_loss_alpha=15.0, bg_loss_n=2
)

hist_cascade = trainer_cascade.fit(train_loader, test_loader, epochs=5)

In [ ]:
# 3. Визуализация
print("\nМатрица ошибок Каскада:")

In [ ]:
plot_confusion_matrices(cascade_model, cascade_model, test_loader, test_dataset, device)

In [ ]:
visualize_classic_gradcam(cascade_model, test_dataset, device, "CascadeNet Grad-CAM")